In [61]:
import pandas as pd
from collections import defaultdict
import json
import numpy as np

In [76]:
# prepare app_metadata.json and app_categories.json
# app_metadata.json: item_id, app_package, app_name, item_id, price, avg_rating, num_reviews, app_categories
# app_categories.json: app_category_id, description, item_ids
app_categories_raw = pd.read_parquet('metadata/app_categories.parquet')
app_item_id_map_raw = pd.read_parquet('metadata/app_item_id_map.parquet')
app_metadata_raw = pd.read_parquet('metadata/app_metadata.parquet')

app_metadata = dict()
app_categories = dict()

# print(app_categories_raw.head())
# print(app_item_id_map_raw.head())
# print(app_metadata_raw.tail())

app_package2item_id = dict()
item_id2app_package = dict()

for row in app_item_id_map_raw.itertuples():
    app_package2item_id[row.app_package] = row.item_id
    item_id2app_package[row.item_id] = row.app_package

category2item_ids = defaultdict(list)
category_description2app_category_id = dict()

for row in app_categories_raw.itertuples():
    category_description2app_category_id[row.category_description] = row.app_category_id

for row in app_metadata_raw.itertuples():
    if row.app_package not in app_package2item_id:
        raise KeyError(f'could not find item_id for app_package: {row.app_package}')
    item_id = app_package2item_id.get(row.app_package)
    category2item_ids[row.app_category].append(item_id)

    app_metadata[item_id] = {
        "item_id": item_id,
        "app_id": row.app_package,
        "app_name": row.app_name,
        "price": 0 if row.price.strip() in ["Install", "not found"] else float(row.price.split()[0][1:]),
        "num_reviews": int(row.num_reviews.replace(",", "")),
        "avg_rating": np.round(float(row.avg_rating) / 5, 4),
        "category_ids": [category_description2app_category_id.get(row.app_category)] if row.app_category in category_description2app_category_id else [],
    }

for row in app_categories_raw.itertuples():
    app_categories[row.category_description] = {
        "app_category_id": row.app_category_id,
        "description": row.category_description,
        "item_ids": category2item_ids.get(row.category_description, [])
    }

with open("metadata/app_metadata.json", "w") as file:
    json.dump(app_metadata, file, indent=2)

with open("metadata/app_categories.json", "w") as file:
    json.dump(app_categories, file, indent=2)


In [77]:
# prepare game_metadata.json and game_categories.json
# game_metadata.json: item_id, app_package, app_name, item_id, price, avg_rating, num_reviews, app_categories
# game_categories.json: app_category_id, description, item_ids
game_categories_raw = pd.read_parquet('metadata/game_categories.parquet')
game_item_id_map_raw = pd.read_parquet('metadata/game_item_id_map.parquet')
game_metadata_raw = pd.read_parquet('metadata/game_metadata.parquet')

game_metadata = dict()
game_categories = dict()

# print(game_categories_raw.head())
# print(game_item_id_map_raw.head())
# print(game_metadata_raw.tail())

app_package2item_id = dict()
item_id2app_package = dict()
item_id2app_name = dict()

for row in game_item_id_map_raw.itertuples():
    app_package2item_id[row.app_package] = row.item_id
    item_id2app_package[row.item_id] = row.app_package
    item_id2app_name[row.item_id] = row.app_name

category2item_ids = defaultdict(list)
app_category_2app_category_id = dict()

for row in game_categories_raw.itertuples():
    app_category_2app_category_id[row.app_category] = row.app_category_id

for row in game_metadata_raw.itertuples():
    if row.id not in app_package2item_id:
        raise KeyError(f'could not find item_id for app_package: {row.app_package}')
    item_id = app_package2item_id.get(row.id)
    app_name = item_id2app_name.get(item_id)
    for category in row.app_category:
        category2item_ids[category].append(item_id)

    game_metadata[item_id] = {
        "item_id": int(item_id),
        "app_id": int(row.id),
        "app_name": app_name,
        "price": float(row.price),
        "num_reviews": int(row.num_reviews),
        "avg_rating": float(row[3]),
        "category_ids": list(map(lambda x: app_category_2app_category_id.get(x), list(row.app_category))),
    }

for row in game_categories_raw.itertuples():
    game_categories[row.category_description] = {
        "app_category_id": row.app_category_id,
        "description": row.category_description,
        "item_ids": category2item_ids.get(row.app_category, [])
    }

with open("metadata/game_metadata.json", "w") as file:
    json.dump(game_metadata, file, indent=2)

with open("metadata/game_categories.json", "w") as file:
    json.dump(game_categories, file, indent=2)
